Outline
This notebook demonstrates how to build a pipeline to predict results using Optuna hyperparameter lookup.

Preprocess and feature Engineering is base on previous notebook.

EDA link: [Baseline XGB of S6E3](https://www.kaggle.com/code/wesleyhuan/baseline-xgb-season-6-episode-3)

Key point: 
1. Data have a lot of non-numerical data in it. This requires to use OrdinalEncoder, OneHotEncoder and LabelEncoder depend on the data type.(Check out the link of **EDA of S6E3**)
2. This is a classification problem, but the submission we returned looks like this. This means we're trying to predict the probability of Churn. Therefore, you might need to use a different evaluation metric first, such as mean squared error (MSE) instead of the area under the curve (AUC).

Submission:

id, Churn

594194, 0.1

594195, 0.3

594196, 0.2

etc.

Steps:

1. Load Data
2. Preprocess Data
3. Train model with Hyper parameter (OPTUNA)
4. Submission

# Load Data And Import Library

In [1]:
# import necessary library
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OrdinalEncoder,LabelEncoder
from sklearn.preprocessing import OneHotEncoder,StandardScaler

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

#for optuna pipline
import optuna
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.base import BaseEstimator, TransformerMixin
import torch

#deal with gender ratio prolem 2026/02/28
from sklearn.model_selection import StratifiedKFold

import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
# config
class CFG:
    train_csv = '/kaggle/input/competitions/playground-series-s6e3/train.csv'
    test_csv = '/kaggle/input/competitions/playground-series-s6e3/test.csv'
    sample_submission_csv = '/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv'
    N_FOLDS = 5
    RANDOM_SEED = 42
    
#torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
train = pd.read_csv(CFG.train_csv)
test = pd.read_csv(CFG.test_csv)

In [4]:
train.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [5]:
test.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,594194,Female,0,Yes,No,72,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Electronic check,115.55,8061.50
1,594195,Female,0,Yes,No,71,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),19.80,1336.50
2,594196,Male,0,No,No,12,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),55.55,633.55
3,594197,Male,0,Yes,Yes,71,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,Two year,No,Credit card (automatic),84.10,6457.15
4,594198,Female,0,No,No,15,Yes,No,Fiber optic,Yes,No,No,No,Yes,Yes,Month-to-month,No,Electronic check,90.35,1233.65


# Preprocess Data

In [6]:
class Preprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.medians = {}
        self.one_hot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')
        self.binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
        self.replace_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                             'TechSupport', 'StreamingTV', 'StreamingMovies']
        self.contract_mapping = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
        self.ohe_cols = ['InternetService', 'PaymentMethod']

    def create_interaction_features(self, df):
        df = df.copy()
        if 'TotalCharges' in df.columns:
            df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
        if 'MonthlyCharges' in df.columns and 'TotalCharges' in df.columns:
            df['Monthly_Fee_Ratio'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1e-6)
        return df

    def fit(self, X, y=None):
        X_tmp = self.create_interaction_features(X)
        curr_num_cols = X_tmp.select_dtypes(include=[np.number]).columns.tolist()
        for col in curr_num_cols:
            self.medians[col] = X_tmp[col].median()
        self.one_hot_encoder.fit(X_tmp[self.ohe_cols].astype(str))
        return self

    def transform(self, X):
        X = X.copy()
        X = self.create_interaction_features(X)
        if 'id' in X.columns: X = X.drop(columns=['id'])
        
        # Binary & Replace logic
        for col in self.binary_cols:
            if col in X.columns:
                X[col] = X[col].apply(lambda x: 1 if x in ['Yes', 'Male'] else 0)
        for col in self.replace_cols:
            if col in X.columns:
                X[col] = X[col].replace({'No internet service': 'No', 'No phone service': 'No'})
                X[col] = X[col].apply(lambda x: 1 if x == 'Yes' else 0)
        
        if 'Contract' in X.columns:
            X['Contract'] = X['Contract'].map(self.contract_mapping).fillna(0).astype(int)

        # Fill NA & OHE
        for col, val in self.medians.items():
            if col in X.columns: X[col] = X[col].fillna(val)

        encoded_array = self.one_hot_encoder.transform(X[self.ohe_cols].astype(str))
        encoded_cols = self.one_hot_encoder.get_feature_names_out(self.ohe_cols)
        encoded_df = pd.DataFrame(encoded_array, columns=encoded_cols, index=X.index)
        
        X = pd.concat([X.drop(columns=self.ohe_cols), encoded_df], axis=1)
        # 移除目標欄位避免洩漏 (如果有的話)
        if 'Churn' in X.columns: X = X.drop(columns=['Churn'])
        return X

In [7]:
preprocessor = Preprocessor()

train_raw = train
X_train_raw = train.drop(columns=['id'])
y_train = train['Churn'].map({'No': 0, 'Yes': 1}).astype(int)

# learn the rule (median value...)
preprocessor.fit(X_train_raw)

# preprocess data
X_train_processed = preprocessor.transform(X_train_raw)

In [8]:
X_train_processed.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,Contract,PaperlessBilling,MonthlyCharges,TotalCharges,Monthly_Fee_Ratio,InternetService_Fiber optic,InternetService_No,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,0,1,1,29,1,0,1,0,1,...,1,1,60.10,1653.85,0.036339,0.0,0.0,0.0,0.0,1.0
1,1,0,1,1,58,1,0,1,1,0,...,2,0,69.50,3778.20,0.018395,0.0,0.0,1.0,0.0,0.0
2,1,0,1,0,58,1,1,0,1,0,...,0,1,100.40,5841.35,0.017188,1.0,0.0,0.0,1.0,0.0
3,0,0,0,0,1,1,0,0,0,0,...,0,1,69.70,70.70,0.985856,1.0,0.0,0.0,1.0,0.0
4,0,0,0,0,1,1,0,0,0,0,...,0,1,70.45,70.45,1.000000,1.0,0.0,0.0,1.0,0.0


In [9]:
y_train.head()

0    0
1    0
2    0
3    1
4    1
Name: Churn, dtype: int64

# Train model with Hyper parameter (OPTUNA)

In [10]:
# 2. 修改 Optuna Objective 使用 Classifier 與 AUC
def XGB_objective(trial):
    model_params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 1500),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
    }
    model_params["scale_pos_weight"] = trial.suggest_float("scale_pos_weight", 1.0, 5.0)
    
    pipeline = Pipeline([
        ("preprocess", Preprocessor()),
        ("model", xgb.XGBClassifier(
            **model_params,
            objective="binary:logistic",
            random_state=CFG.RANDOM_SEED,
            n_jobs=-1
        ))
    ])

    # 使用 StratifiedKFold 保持 1:4 的比例
    skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.RANDOM_SEED)
    
    # 競賽通常看 roc_auc
    scores = cross_val_score(pipeline, X_train_raw, y_train, cv=skf, scoring="roc_auc")
    return scores.mean()

In [11]:
# use optuna to find best parameter
study = optuna.create_study(direction="maximize")
study.optimize(XGB_objective, n_trials=30)

[I 2026-03-29 12:06:30,931] A new study created in memory with name: no-name-2f60b4df-0202-4e34-b0de-a9690543bcb6
[I 2026-03-29 12:08:50,171] Trial 0 finished with value: 0.9149889254900163 and parameters: {'n_estimators': 702, 'max_depth': 5, 'learning_rate': 0.016423100694603707, 'subsample': 0.7727769582316092, 'colsample_bytree': 0.6216449978145613, 'scale_pos_weight': 3.1885526112982574}. Best is trial 0 with value: 0.9149889254900163.
[I 2026-03-29 12:12:13,994] Trial 1 finished with value: 0.9160283504789319 and parameters: {'n_estimators': 1020, 'max_depth': 7, 'learning_rate': 0.02908438719744598, 'subsample': 0.7300957484611436, 'colsample_bytree': 0.7812040419228194, 'scale_pos_weight': 4.710327853645179}. Best is trial 1 with value: 0.9160283504789319.
[I 2026-03-29 12:14:33,773] Trial 2 finished with value: 0.9144878196422154 and parameters: {'n_estimators': 570, 'max_depth': 8, 'learning_rate': 0.09265743625715417, 'subsample': 0.8376241595302608, 'colsample_bytree': 0.79

In [12]:
print(f"Best AUC score: {study.best_value:.4f}")
print("Best parameters:", study.best_params)

Best AUC score: 0.9164
Best parameters: {'n_estimators': 1453, 'max_depth': 4, 'learning_rate': 0.0531683480048988, 'subsample': 0.730302959646642, 'colsample_bytree': 0.6634360140876122, 'scale_pos_weight': 3.6847668503715862}


# Submission

In [13]:
final_model = Pipeline([
    ("preprocess", Preprocessor()),
    ("model", xgb.XGBClassifier(**study.best_params, objective="binary:logistic"))
])

final_model.fit(X_train_raw, y_train)
test_id = test["id"]
test = test.drop(columns=['id'])
y_prob = final_model.predict_proba(test)[:, 1]

In [14]:
submission = pd.DataFrame({'id': test_id, 'Churn': y_prob})
submission.to_csv('submission.csv', index=False)

In [15]:
submission.head()

,id,Churn
0,594194,0.283633
1,594195,0.000964
2,594196,0.291507
3,594197,0.013185
4,594198,0.820199
